# Clasificación Inteligente de Correos con Transformers

## Objetivo del Proyecto
Desarrollar un sistema de clasificación automática de correos electrónicos para detectar:

- **Sentimiento** (positivo, neutro, negativo)
- **Prioridad** (alta, media, baja)
- **Categoría** (queja, solicitud, comercial, otro)

Este sistema permite priorizar y gestionar correos de forma más eficiente.

---

## Herramientas y Tecnologías

| Herramienta            | Descripción                                                                 |
|------------------------|-----------------------------------------------------------------------------|
| **Python**             | Lenguaje principal del proyecto                                             |
| **Hugging Face Transformers** | Librería para usar modelos preentrenados del tipo BERT/RoBERTa               |
| **RoBERTuito**         | Modelo preentrenado tipo RoBERTa optimizado para el idioma español         |
| **Flask**              | Framework para exponer el modelo como una aplicación web                   |
| **Gmail API**          | Conexión a una cuenta real de correo para extraer y analizar emails        |

---

## ¿Qué modelo hemos usado?

- Utilizamos `pysentimiento/robertuito-base-uncased` (de Hugging Face), una versión ligera de **RoBERTa** entrenada en español.
- RoBERTa es una arquitectura basada en **BERT**, pero con mejoras en rendimiento y entrenamiento.
- Aplicamos *fine-tuning* para adaptar el modelo a nuestro propio dataset de correos electrónicos clasificados.

---

## Entrenamiento personalizado (Fine-tuning)

- Concatenamos **asunto + cuerpo del correo** como entrada al modelo.
- Creamos una estructura estándar para cada modelo (`sentiment`, `priority`, `category`).
- Entrenamos con `Trainer` de Hugging Face, usando validación y **early stopping**.
- Evaluamos con métricas de **accuracy** y **F1-score**.
- Guardamos el modelo y su tokenizer para usarlo en producción.



## CREACIÓN DEL MODELO PARA CLASIFICAR POR SENTIMIENTO

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np
import torch
from transformers import EarlyStoppingCallback

df = pd.read_csv("mails_dataset.csv") 

df['text_combined'] = df['subject'].fillna('') + ' </s> ' + df['text'].fillna('')

# Nos centramos en la columna combinada y la etiqueta sentiment
df = df[['text_combined', 'sentiment']].dropna()

# Codificar etiquetas
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["sentiment"])

# Guardar las etiquetas para decodificar luego
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}
id2label = {i: label for label, i in label2id.items()}
print(label_encoder.classes_)

#Crear Dataset de Hugging Face
dataset = Dataset.from_pandas(df.rename(columns={"text_combined": "text", "label": "label"}))

#Tokenización
model_name = "pysentimiento/robertuito-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

dataset = dataset.map(tokenize, batched=True)

#División entrenamiento y validación
dataset = dataset.train_test_split(test_size=0.2)

#Cargar modelo preentrenado
num_labels = len(label2id)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

#Configurar entrenamiento
training_args = TrainingArguments(
    output_dir="./sentiment_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,      
    metric_for_best_model="f1",        
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    logging_dir="./logs_sentiment",
    logging_steps=10
)


#Función de evaluación
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

#Entrenador
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

#Entrenar
trainer.train()

#Guardar el modelo y tokenizer
trainer.save_model("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")


['negativo' 'neutro' 'positivo']


c:\Users\Aaron\OneDrive\Escritorio\Proyecto\MailClassifier\.venv\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/323 [00:00<?, ?B/s]

c:\Users\Aaron\OneDrive\Escritorio\Proyecto\MailClassifier\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aaron\.cache\huggingface\hub\models--pysentimiento--robertuito-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json:   0%|          | 0.00/858k [00:00<?, ?B/s]

...

c:\Users\Aaron\OneDrive\Escritorio\Proyecto\MailClassifier\.venv\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.1261, 'grad_norm': 1.5805784463882446, 'learning_rate': 1.944444444444445e-06, 'epoch': 5.42}
{'loss': 0.0784, 'grad_norm': 1.405696153640747, 'learning_rate': 5.555555555555555e-07, 'epoch': 5.83}


  0%|          | 0/6 [00:00<?, ?it/s]

{'eval_loss': 0.2898481786251068, 'eval_accuracy': 0.9148936170212766, 'eval_f1': 0.915897435897436, 'eval_runtime': 5.6762, 'eval_samples_per_second': 8.28, 'eval_steps_per_second': 1.057, 'epoch': 6.0}
{'train_runtime': 626.9829, 'train_samples_per_second': 1.79, 'train_steps_per_second': 0.23, 'train_loss': 0.3683791578643852, 'epoch': 6.0}


('./sentiment_model\\tokenizer_config.json',
 './sentiment_model\\special_tokens_map.json',
 './sentiment_model\\tokenizer.json')

In [2]:
from sklearn.metrics import classification_report
import numpy as np

# Obtener predicciones sobre el set de evaluación
predictions = trainer.predict(dataset["test"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

# Mostrar el reporte con los nombres reales de las clases
print("Reporte de clasificación por clase:")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))


c:\Users\Aaron\OneDrive\Escritorio\Proyecto\MailClassifier\.venv\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/6 [00:00<?, ?it/s]

Reporte de clasificación por clase:
              precision    recall  f1-score   support

    negativo       0.85      0.92      0.88        12
      neutro       0.87      0.87      0.87        15
    positivo       1.00      0.95      0.97        20

    accuracy                           0.91        47
   macro avg       0.90      0.91      0.91        47
weighted avg       0.92      0.91      0.92        47

